<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/QE_JARVIS_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda --version

In [ ]:
import condacolab
condacolab.check()

In [ ]:
! conda install qe jarvis-tools -y

In [ ]:
!which pw.x

In [ ]:
aluminum="""FCC Al
1.0
4.06741 0.0 0.0
0.0 4.06741 0.0
0.0 0.0 4.06741
Al
4
direct
0.0 0.0 0.0
0.0 0.5 0.5
0.5 0.0 0.5
0.5 0.5 0.0
"""
# aluminum="""Al
# 1.0
# 2.4907700981617955 0 1.4380466239515413
# 0.8302566980301707 2.348320626706396 1.438046623951541
# 0 0 2.8760942620064256
# Al
# 1
# Cartesian
# 0.0 0.0 0.0
# """

In [ ]:
from jarvis.core.atoms import Atoms
from jarvis.io.vasp.inputs import Poscar
atoms = Poscar.from_string(aluminum).atoms
atoms

In [ ]:
%%time
qe_cmd = '/usr/local/bin/pw.x'
!rm relax.json
from jarvis.tasks.qe.qe import QEjob
from jarvis.core.kpoints import Kpoints3D as Kpoints
kp=Kpoints([[5,5,5]])
# For better quality electronic structure, try k-points of [4,4,5] and also try to include DFT+U corrections
relax = {
            "control": {
                #"calculation": "'scf'",
                "calculation":  "'vc-relax'",
                "restart_mode": "'from_scratch'",
                "prefix": "'RELAX'",
                "outdir": "'./'",
                "tstress": ".true.",
                "tprnfor": ".true.",
                "disk_io": "'nowf'",
                "wf_collect": ".true.",
                "pseudo_dir": None,
                "verbosity": "'high'",
                "nstep": 100,
            },
            "system": {
                "ibrav": 0,
                "nat": None,
                "ntyp": None,
                "ecutwfc": 45,
                "ecutrho": 250,
                "q2sigma": 1,
                "ecfixed": 44.5,
                "qcutz": 800,
                "occupations": "'smearing'",
                "degauss": 0.01,
                "lda_plus_u": ".false.",
            },
            "electrons": {
                "diagonalization": "'david'",
                "mixing_mode": "'local-TF'",
                "mixing_beta": 0.3,
                "conv_thr": "1d-9",
            },
            "ions": {"ion_dynamics": "'bfgs'"},
            "cell": {"cell_dynamics": "'bfgs'", "cell_dofree": "'all'"},
        }
qejob_relax = QEjob(
    atoms=atoms,
    input_params=relax,
    output_file="relax.out",
    qe_cmd=qe_cmd,
    jobname="relax",
    kpoints=kp,
    input_file="arelax.in",
    url=None,
    psp_dir=None,
    psp_temp_name=None,
)
info = qejob_relax.runjob()

In [ ]:
info

In [ ]:
from jarvis.io.qe.outputs import DataFileSchema
xml_dat = DataFileSchema(filename=info['xml_path'])
final_strt = xml_dat.final_structure
initial_stress = xml_dat.stress
initial_stress

In [ ]:
final_strt

In [ ]:
%matplotlib inline
energies,DOS = dos = schema.dos()
import matplotlib.pyplot as plt
plt.plot(energies,DOS)
plt.xlabel('E-(eV)')
plt.ylabel('Density of states (arb. unit)')
plt.fill_between(energies, 0, DOS, where=(energies < 1), facecolor='blue', alpha=0.25)
plt.axvline(x=0,linestyle='-.',color='green')

plt.xlim([-5,5])
plt.ylim([0,5])
plt.show()

In [ ]:
strain=0.01
#strained_atoms=j_atoms.strain_atoms([[strain,0,0],[0,1,0],[0,1,0]]).ase_converter()
strained_atoms=final_strt.strain_atoms([strain,0,0])

In [ ]:
strained_atoms

In [ ]:
%%time
qe_cmd = '/usr/local/bin/pw.x'
!rm relax.json
from jarvis.tasks.qe.qe import QEjob
from jarvis.core.kpoints import Kpoints3D as Kpoints
kp=Kpoints([[5,5,5]])
# For better quality electronic structure, try k-points of [4,4,5] and also try to include DFT+U corrections
relax = {
            "control": {
                "calculation": "'scf'",
                #"calculation":  "'vc-relax'",
                "restart_mode": "'from_scratch'",
                "prefix": "'RELAX'",
                "outdir": "'./'",
                "tstress": ".true.",
                "tprnfor": ".true.",
                "disk_io": "'nowf'",
                "wf_collect": ".true.",
                "pseudo_dir": None,
                "verbosity": "'high'",
                "nstep": 100,
            },
            "system": {
                "ibrav": 0,
                "nat": None,
                "ntyp": None,
                "ecutwfc": 45,
                "ecutrho": 250,
                "q2sigma": 1,
                "ecfixed": 44.5,
                "qcutz": 800,
                "occupations": "'smearing'",
                "degauss": 0.01,
                "lda_plus_u": ".false.",
            },
            "electrons": {
                "diagonalization": "'david'",
                "mixing_mode": "'local-TF'",
                "mixing_beta": 0.3,
                "conv_thr": "1d-9",
            },
            "ions": {"ion_dynamics": "'bfgs'"},
            "cell": {"cell_dynamics": "'bfgs'", "cell_dofree": "'all'"},
        }
qejob_relax = QEjob(
    atoms=strained_atoms,
    input_params=relax,
    output_file="relax.out",
    qe_cmd=qe_cmd,
    jobname="relax",
    kpoints=kp,
    input_file="arelax.in",
    url=None,
    psp_dir=None,
    psp_temp_name=None,
)
info_strained = qejob_relax.runjob()

In [ ]:
from jarvis.io.qe.outputs import DataFileSchema
xml_dat_strained = DataFileSchema(filename=info_strained['xml_path'])

final_stress = xml_dat_strained.stress
final_stress

In [ ]:
160*(final_stress-initial_stress)/strain

In [ ]:
strain=0.01
strained_atoms=final_strt.strain_atoms([[0,0,0],[0,strain,0],[0,0,0]])
strained_atoms


In [ ]:
final_strt

In [ ]:
%%time
qe_cmd = '/usr/local/bin/pw.x'
!rm strained.json
from jarvis.tasks.qe.qe import QEjob
from jarvis.core.kpoints import Kpoints3D as Kpoints
kp=Kpoints([[9,9,9]])
# For better quality electronic structure, try k-points of [4,4,5] and also try to include DFT+U corrections
relax = {
            "control": {
                "calculation": "'scf'",
                #"calculation":  "'vc-relax'",
                "restart_mode": "'from_scratch'",
                "prefix": "'RELAX'",
                "outdir": "'./'",
                "tstress": ".true.",
                "tprnfor": ".true.",
                "disk_io": "'nowf'",
                "wf_collect": ".true.",
                "pseudo_dir": None,
                "verbosity": "'high'",
                "nstep": 100,
            },
            "system": {
                "ibrav": 0,
                "nat": None,
                "ntyp": None,
                "ecutwfc": 45,
                "ecutrho": 250,
                "q2sigma": 1,
                "ecfixed": 44.5,
                "qcutz": 800,
                "occupations": "'smearing'",
                "degauss": 0.01,
                "lda_plus_u": ".false.",
            },
            "electrons": {
                "diagonalization": "'david'",
                "mixing_mode": "'local-TF'",
                "mixing_beta": 0.3,
                "conv_thr": "1d-9",
            },
            "ions": {"ion_dynamics": "'bfgs'"},
            "cell": {"cell_dynamics": "'bfgs'", "cell_dofree": "'all'"},
        }
qejob_relax = QEjob(
    atoms=strained_atoms,
    input_params=relax,
    output_file="relax.out",
    qe_cmd=qe_cmd,
    jobname="strained",
    kpoints=kp,
    input_file="arelax.in",
    url=None,
    psp_dir=None,
    psp_temp_name=None,
)
info_strained = qejob_relax.runjob()

In [ ]:
!ls -altr

In [ ]:
from jarvis.io.qe.outputs import DataFileSchema
xml_dat_strained = DataFileSchema(filename=info_strained['xml_path'])

final_stress = xml_dat_strained.stress
final_stress

In [ ]:
160*(final_stress-initial_stress)/strain

In [ ]:
!which pw.x

In [ ]:
%%time
from jarvis.tasks.qe.converg import converg_kpoints
kp_len = converg_kpoints(atoms,qe_cmd='/usr/local/bin/pw.x')

In [ ]:
pip install mechelastic

In [ ]:
!wget https://raw.githubusercontent.com/romerogroup/MechElastic/master/examples/QE_ElaStic/scf.in

In [ ]:
!pw.x < scf.in

In [ ]:

from mechelastic.parsers import QE_ElaStic_Parser
from mechelastic import calculate_elastic
from mechelastic.core import ElasticProperties

calculate_elastic(
    qe_outfile="ElaStic_2nd.out", qe_infile="scf.in", crystal="cubic", code="qe_ElaStic"
)


parser = QE_ElaStic_Parser(outfile="ElaStic_2nd.out", infile="scf.in")


elastic_tensor = parser.elastic_tensor
structure = parser.structure
lattice_constant = parser.lattice_constant
crystal_type = "cubic"

elastic_properties = ElasticProperties(elastic_tensor, structure, crystal_type)

bulk_modulus_voigt = elastic_properties.bulk_modulus_voigt